In [0]:
%run /Workspace/F1_project/app_lib/0.ALDS_access_setup

In [0]:
%run /Workspace/F1_project/app_lib/0.1.configs_lib

In [0]:
dbutils.widgets.text("p_csv_filepath", "")
dbutils.widgets.text("p_csv_filename", "")
dbutils.widgets.text("p_csv_schema", "")
dbutils.widgets.text("p_csv_header", "")
dbutils.widgets.text("p_transformations_file", "")
dbutils.widgets.text("p_partitions", "")

v_csv_filepath = dbutils.widgets.get("p_csv_filepath")
v_csv_filename = dbutils.widgets.get("p_csv_filename")
v_csv_schema = dbutils.widgets.get("p_csv_schema")
v_csv_header = dbutils.widgets.get("p_csv_header").lower() == "true"
v_tx = dbutils.widgets.get("p_transformations_file")
v_transformations_file = "/Workspace/F1_project/custom_transformation/" + v_tx + ".ipynb"
v_partitions = dbutils.widgets.get("p_partitions")




In [0]:
csv_df = spark.read\
        .csv(f"{v_csv_filepath}", header=v_csv_header, schema=v_csv_schema)
       

In [0]:
if v_tx != "":
    %run $v_transformations_file 

In [0]:
if v_tx != "":
    csv_df = transformation(csv_df)
    print("Transformations applied")
else:
    print("No transformations specified")

In [0]:
csv_df = populate_ingestion_time(csv_df)

In [0]:

part_cols = v_partitions.split(",") if v_partitions else []

writer = csv_df.write.mode("overwrite")

if part_cols:
    writer = writer.partitionBy(*part_cols)

writer.parquet(f"{processed_folder}/{v_csv_filename}")

In [0]:
display(csv_df)

In [0]:
display(spark.read.parquet(f"{processed_folder}/{v_csv_filename}")) 

In [0]:
dbutils.notebook.exit(f"{v_csv_filename} has been ingested into processed container")